# Expert Atlas — Colab Demonstration Notebook

**⚠️ IMPORTANT: This is a method demonstration on a reduced subset (~40 prompts).**

- The full Expert Atlas run used **480 prompts** and took **~12.9 hours on CPU**.
- This notebook runs on **~40 prompts** (a subset of the factorial design) to demonstrate the complete pipeline in **~10–15 minutes on a free T4 GPU**.
- **It will NOT reproduce the exact numbers in `docs/FINDINGS.md`** — those require the full 480-prompt capture and analysis.
- The purpose is to show that the pipeline works end-to-end and to let you inspect the method on real model outputs.

For the real results, see:
- `docs/FINDINGS.md` — H1–H6 verdicts with effect-size filtering
- `docs/ORTHOGONALITY.md` — routing orthogonality analysis
- `docs/ABLATION.md` — causal ablation test
- `docs/METHOD.md` — full statistical method (this notebook implements it)

---

## What this notebook does

1. **Installs dependencies** (torch, transformers, numpy, scipy, statsmodels, umap-learn, pyarrow, pyyaml)
2. **Loads OLMoE-1B-7B-0924** in bfloat16 on GPU
3. **Captures router decisions** on a subset of the factorial probe set (~40 prompts)
4. **Computes lift matrices** with equal-token-budget subsampling
5. **Runs significance testing** (chi-squared + Benjamini–Hochberg FDR)
6. **Applies the effect-size filter** (|lift| ≥ 1.0 = ≥2× fold change)
7. **Reports H6 split-half replication** (the project gate)
8. **Shows top specialists** per topic with lift values
9. **Renders a 3-D UMAP atlas** of experts coloured by co-activation community (inline via py3Dmol fallback or static matplotlib)

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────
%pip install -q torch transformers accelerate numpy scipy statsmodels umap-learn pyarrow pyyaml matplotlib

# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 2. Load model and probe set ──────────────────────────────────────────
import json
import yaml
from pathlib import Path
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "allenai/OLMoE-1B-7B-0924"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16

print(f"Loading {MODEL_ID} on {DEVICE} in {DTYPE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    output_router_logits=True,
    device_map="auto" if DEVICE == "cuda" else None,
)
model.eval()

# Model shape
n_layers = model.config.num_hidden_layers
n_experts = model.config.num_experts
top_k = model.config.num_experts_per_tok
norm_topk_prob = getattr(model.config, "norm_topk_prob", False)
print(f"Model: {n_layers} layers × {n_experts} experts, top-{top_k}, norm_topk_prob={norm_topk_prob}")

# Load probe set (subset for demo)
PROBE_SET_PATH = Path("probes/probe_set_v1.yaml")
if not PROBE_SET_PATH.exists():
    # If not in repo, fetch from GitHub
    import urllib.request
    url = "https://raw.githubusercontent.com/manjunathbhaskar/expert-atlas/main/probes/probe_set_v1.yaml"
    print(f"Fetching probe set from {url}...")
    PROBE_SET_PATH.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(url, PROBE_SET_PATH)

probe_set = yaml.safe_load(PROBE_SET_PATH.read_text())
prompts = probe_set["prompts"]

# DEMO SUBSET: pick ~40 prompts balanced across topics, using split=A only
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

by_topic = {}
for p in prompts:
    if p.get("split") == "A":
        by_topic.setdefault(p["topic"], []).append(p)

demo_prompts = []
for topic, plist in by_topic.items():
    demo_prompts.extend(random.sample(plist, min(4, len(plist))))  # 4 per topic × 10 topics = 40

print(f"Demo subset: {len(demo_prompts)} prompts across {len(by_topic)} topics (split=A only)")
for p in demo_prompts:
    print(f"  [{p['topic']}/{p['lang']}/{p['register']}/{p['format']}] {p['text'][:60]}...")

In [ ]:
# ── 3. Capture router traces ─────────────────────────────────────────────

def route_from_logits(router_logits: torch.Tensor, top_k: int, norm_topk_prob: bool):
    """Reproduce the model's top-k selection from raw logits."""
    routing_weights = torch.softmax(router_logits, dim=-1, dtype=torch.float32)
    topk_weights, topk_ids = torch.topk(routing_weights, top_k, dim=-1)
    topk_mass = topk_weights.sum(dim=-1)
    if norm_topk_prob:
        topk_weights = topk_weights / topk_mass.unsqueeze(-1)
    return topk_ids, topk_weights, topk_mass

traces = []  # list of dicts: prompt_id, token_pos, token_id, layer, expert_ids, gate_weights, topk_mass

for prompt in demo_prompts:
    prompt_id = prompt["prompt_id"]
    text = prompt["text"]
    
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    input_ids = inputs["input_ids"][0]
    
    with torch.no_grad():
        out = model(**inputs, output_router_logits=True)
    
    router_logits = out.router_logits  # tuple of (n_tokens, n_experts) per layer
    
    for layer, logits in enumerate(router_logits):
        expert_ids, gate_weights, topk_mass = route_from_logits(logits, top_k, norm_topk_prob)
        n_tokens = input_ids.shape[0]
        for t in range(n_tokens):
            traces.append({
                "prompt_id": prompt_id,
                "token_pos": t,
                "token_id": int(input_ids[t]),
                "layer": layer,
                "expert_ids": expert_ids[t].tolist(),
                "gate_weights": [float(w) for w in gate_weights[t].tolist()],
                "topk_mass": float(topk_mass[t]),
            })
    
    if len(traces) % 5000 == 0:
        print(f"  captured {len(traces)} token×layer records...")

print(f"Total trace records: {len(traces)}")
print(f"Unique tokens: {len(set((r['prompt_id'], r['token_pos']) for r in traces))}")

In [ ]:
# ── 4. Build count matrices with equal token budget ──────────────────────

from collections import defaultdict

FACTORS = ("topic", "lang", "register", "format")

def cell_key(prompt):
    return tuple(prompt[f] for f in FACTORS)

prompts_by_id = {p["prompt_id"]: p for p in demo_prompts}

# Group tokens by factorial cell
tokens_by_cell = defaultdict(set)
for r in traces:
    key = cell_key(prompts_by_id[r["prompt_id"]])
    tokens_by_cell[key].add((r["prompt_id"], r["token_pos"]))

# Equal token budget = minimum across cells
budget = min(len(v) for v in tokens_by_cell.values())
print(f"Tokens per cell (budget): {budget}")
print(f"Cells: {len(tokens_by_cell)}")

# Subsample
rng = np.random.default_rng(0)
kept_tokens = set()
for key, toks in tokens_by_cell.items():
    ordered = sorted(toks)
    if len(ordered) <= budget:
        kept_tokens.update(ordered)
    else:
        idx = rng.choice(len(ordered), size=budget, replace=False)
        kept_tokens.update(ordered[i] for i in idx)

kept_traces = [r for r in traces if (r["prompt_id"], r["token_pos"]) in kept_tokens]
print(f"Kept {len(kept_traces)} records ({len(kept_tokens)} distinct tokens)")

# Aggregate per factor
def aggregate(kept_traces, prompts_by_id, domain_factor):
    domains = sorted({p[domain_factor] for p in prompts_by_id.values()})
    d_index = {d: i for i, d in enumerate(domains)}
    n_experts_total = n_layers * n_experts
    counts = np.zeros((n_experts_total, len(domains)), dtype=np.float64)
    
    for r in kept_traces:
        col = d_index[prompts_by_id[r["prompt_id"]][domain_factor]]
        base = int(r["layer"]) * n_experts
        for e in r["expert_ids"]:
            counts[base + int(e), col] += 1.0
    
    uids = [f"L{l:02d}E{e:02d}" for l in range(n_layers) for e in range(n_experts)]
    return counts, uids, domains

count_matrices = {}
for f in FACTORS:
    counts, uids, domains = aggregate(kept_traces, prompts_by_id, f)
    count_matrices[f] = {"counts": counts, "uids": uids, "domains": domains}
    print(f"  {f}: {counts.shape[0]} experts × {counts.shape[1]} domains, total tokens={int(counts.sum())}")

In [ ]:
# ── 5. Compute lift, significance, FDR, effect size ──────────────────────

from scipy.stats import chi2 as chi2_dist
from statsmodels.stats.multitest import multipletests

def compute_lift(counts, laplace=1.0):
    """log2(P(e|d) / P(e)) with Laplace smoothing."""
    counts = np.asarray(counts, dtype=np.float64)
    n_experts, n_domains = counts.shape
    domain_totals = counts.sum(axis=0)
    expert_totals = counts.sum(axis=1)
    grand_total = counts.sum()
    p_e_given_d = (counts + laplace) / (domain_totals[None, :] + laplace * n_experts)
    p_e = (expert_totals + laplace) / (grand_total + laplace * n_experts)
    return np.log2(p_e_given_d / p_e[:, None])

def chi2_pvalues_fast(counts):
    """Vectorised Yates-corrected 2x2 chi-squared per (expert, domain)."""
    counts = np.asarray(counts, dtype=np.float64)
    a = counts
    row = counts.sum(axis=1, keepdims=True)
    col = counts.sum(axis=0, keepdims=True)
    n = counts.sum()
    if n == 0:
        return np.ones_like(counts)
    b = row - a
    c = col - a
    d = n - a - b - c
    num = n * np.clip(np.abs(a * d - b * c) - n / 2.0, 0.0, None) ** 2
    den = row * col * (n - row) * (n - col)
    with np.errstate(divide="ignore", invalid="ignore"):
        stat = np.where(den > 0, num / den, 0.0)
    p = chi2_dist.sf(stat, df=1)
    return np.where(den > 0, p, 1.0)

def bh_fdr(pvalues, q=0.05):
    """Benjamini-Hochberg FDR across the whole flattened matrix."""
    flat = np.asarray(pvalues).flatten()
    reject, _, _, _ = multipletests(flat, alpha=q, method="fdr_bh")
    return reject.reshape(pvalues.shape)

# Process each factor
results = {}
for f in FACTORS:
    cm = count_matrices[f]
    counts = cm["counts"]
    uids = cm["uids"]
    domains = cm["domains"]
    
    lift = compute_lift(counts)
    pvals = chi2_pvalues_fast(counts)
    sig = bh_fdr(pvals, q=0.05)
    
    # Effect size filter: |lift| >= 1.0 (>=2x fold)
    meaningful = sig & (np.abs(lift) >= 1.0)
    
    results[f] = {
        "lift": lift,
        "pvals": pvals,
        "significant": sig,
        "meaningful": meaningful,
        "uids": uids,
        "domains": domains,
    }
    
    n_sig = sig.sum()
    n_meaningful = meaningful.sum()
    print(f"{f}: {n_sig}/{sig.size} FDR-significant ({100*n_sig/sig.size:.1f}%), {n_meaningful} also |lift|≥1.0 ({100*n_meaningful/sig.size:.1f}%)")

In [ ]:
# ── 6. H6 — Split-half replication (the project gate) ────────────────────

from scipy.stats import spearmanr

# We need split B prompts too for H6. For demo, we'll just split our kept tokens randomly.
# Real H6 uses the pre-declared A/B split in the probe set.

def split_half_replication(lift_a, lift_b):
    """Spearman correlation between two lift matrices (flattened)."""
    a = np.asarray(lift_a, dtype=np.float64).ravel()
    b = np.asarray(lift_b, dtype=np.float64).ravel()
    return float(spearmanr(a, b).statistic)

# For demo: random 50/50 split of kept tokens, rebuild counts, compute lift
kept_list = list(kept_tokens)
rng.shuffle(kept_list)
mid = len(kept_list) // 2
split_a_tokens = set(kept_list[:mid])
split_b_tokens = set(kept_list[mid:])

traces_a = [r for r in kept_traces if (r["prompt_id"], r["token_pos"]) in split_a_tokens]
traces_b = [r for r in kept_traces if (r["prompt_id"], r["token_pos"]) in split_b_tokens]

h6_results = {}
for f in FACTORS:
    cm_a = aggregate(traces_a, prompts_by_id, f)
    cm_b = aggregate(traces_b, prompts_by_id, f)
    lift_a = compute_lift(cm_a[0])
    lift_b = compute_lift(cm_b[0])
    rho = split_half_replication(lift_a, lift_b)
    h6_results[f] = rho
    status = "PASS" if rho >= 0.5 else "FAIL"
    print(f"H6 {f}: ρ = {rho:.3f} → {status} (threshold ≥ 0.5)")

In [ ]:
# ── 7. H1 — Per-expert domain affinity (topic factor) ────────────────────

topic_res = results["topic"]
lift = topic_res["lift"]
meaningful = topic_res["meaningful"]
uids = topic_res["uids"]
domains = topic_res["domains"]

# Per expert: does it have ANY domain that is both FDR-sig AND |lift|>=1.0?
expert_has_meaningful = meaningful.any(axis=1)
n_experts_meaningful = expert_has_meaningful.sum()
print(f"H1 (topic): {n_experts_meaningful}/{len(uids)} experts ({100*n_experts_meaningful/len(uids):.1f}%) have ≥1 meaningful domain")
print(f"Falsification bar: <5% → {"PASS" if n_experts_meaningful/len(uids) >= 0.05 else "FAIL"}")

# Top specialists by max lift
max_lift = lift.max(axis=1)
max_lift_domain = [domains[np.argmax(lift[i])] for i in range(lift.shape[0])]
top_idx = np.argsort(max_lift)[-20:][::-1]

print("\nTop 20 specialists (by max lift):")
print(f"{'Expert':<10} {'Max Lift':>8} {'Domain':<15} {'Meaningful':>10}")
for i in top_idx:
    flag = "✓" if expert_has_meaningful[i] else ""
    print(f"{uids[i]:<10} {max_lift[i]:>8.3f} {max_lift_domain[i]:<15} {flag:>10}")

In [ ]:
# ── 8. Show per-domain top experts ───────────────────────────────────────

print("\nPer-domain top 5 experts (lift ≥ 1.0 only):")
for d_idx, domain in enumerate(domains):
    col_lift = lift[:, d_idx]
    col_meaningful = meaningful[:, d_idx]
    top_idx = np.argsort(col_lift)[-5:][::-1]
    print(f"\n{domain}:")
    for i in top_idx:
        if col_lift[i] >= 1.0:
            flag = "✓" if col_meaningful[i] else "(sig only)"
            print(f"  {uids[i]}: lift={col_lift[i]:.3f} {flag}")
        else:
            print(f"  {uids[i]}: lift={col_lift[i]:.3f} (below 2x)")

In [ ]:
# ── 9. 3-D UMAP layout (presentational) ──────────────────────────────────

try:
    import umap
    HAVE_UMAP = True
except ImportError:
    HAVE_UMAP = False

def compute_layout(lift_matrix, method="auto", seed=0, jitter=1e-3):
    """UMAP on lift vectors with PCA fallback."""
    x = np.asarray(lift_matrix, dtype=np.float64)
    
    if method in ("umap", "auto") and HAVE_UMAP:
        try:
            n_neighbors = int(min(15, max(2, x.shape[0] - 1)))
            reducer = umap.UMAP(
                n_components=3, n_neighbors=n_neighbors, min_dist=0.1,
                metric="cosine", random_state=seed,
            )
            coords = reducer.fit_transform(x)
            used = "umap"
        except Exception:
            used = "pca"
    else:
        used = "pca"
    
    if used == "pca":
        xc = x - x.mean(axis=0, keepdims=True)
        u, s, vt = np.linalg.svd(xc, full_matrices=False)
        k = min(3, vt.shape[0])
        comps = vt[:k]
        for i in range(k):
            if comps[i][np.argmax(np.abs(comps[i]))] < 0:
                comps[i] = -comps[i]
        coords = xc @ comps.T
        if coords.shape[1] < 3:
            coords = np.pad(coords, ((0, 0), (0, 3 - coords.shape[1])))
    
    # Scale to ~[-10, 10]
    span = np.ptp(coords, axis=0)
    scale = 10.0 / max(float(span.max()), 1e-9)
    coords = (coords - coords.mean(axis=0, keepdims=True)) * scale
    
    # Deterministic jitter
    rng = np.random.default_rng(seed)
    coords = coords + rng.normal(scale=jitter, size=coords.shape)
    
    return coords, used

coords, method_used = compute_layout(lift, method="auto")
print(f"Layout method: {method_used}")
print(f"Coordinate ranges: X[{coords[:,0].min():.1f}, {coords[:,0].max():.1f}], Y[{coords[:,1].min():.1f}, {coords[:,1].max():.1f}], Z[{coords[:,2].min():.1f}, {coords[:,2].max():.1f}]")

In [ ]:
# ── 10. Inline 3-D scatter (matplotlib) ──────────────────────────────────

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Colour by max lift domain (specialisation)
max_lift = lift.max(axis=1)
max_domain_idx = lift.argmax(axis=1)
domain_colors = plt.cm.tab10(np.arange(len(domains)) % 10)
colors = domain_colors[max_domain_idx]
sizes = 10 + 50 * np.clip(max_lift, 0, 3)  # size by positive lift

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], 
                c=colors, s=sizes, alpha=0.6, edgecolors='none')

# Legend
for i, domain in enumerate(domains):
    ax.scatter([], [], [], c=[domain_colors[i]], label=domain, s=50, alpha=0.6)
ax.legend(title="Max-lift domain", bbox_to_anchor=(1.05, 1), loc='upper left')

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_zlabel("UMAP 3")
ax.set_title(f"Expert Atlas (demo, {method_used}) — coloured by max-lift topic, sized by lift magnitude")
plt.tight_layout()
plt.show()

In [ ]:
# ── 11. Summary ──────────────────────────────────────────────────────────

print("=" * 60)
print("DEMO RUN SUMMARY")
print("=" * 60)
print(f"Model: {MODEL_ID}")
print(f"Prompts captured: {len(demo_prompts)} (of 480 in full run)")
print(f"Tokens per cell (after subsampling): {budget}")
print(f"Total token×layer records: {len(kept_traces)}")
print()
print("H6 (split-half replication, gate):")
for f, rho in h6_results.items():
    print(f"  {f}: ρ = {rho:.3f} {'✓ PASS' if rho >= 0.5 else '✗ FAIL'}")
print()
print("H1 (per-expert meaningful affinity, topic):")
print(f"  {n_experts_meaningful}/{len(uids)} experts ({100*n_experts_meaningful/len(uids):.1f}%) have ≥1 domain with FDR-sig AND |lift|≥1.0")
print(f"  Falsification bar (<5%): {'✓ PASS' if n_experts_meaningful/len(uids) >= 0.05 else '✗ FAIL'}")
print()
print("H3 (factor separability — meaningful rate):")
for f in FACTORS:
    n_sig = results[f]["significant"].sum()
    n_mean = results[f]["meaningful"].sum()
    total = results[f]["significant"].size
    print(f"  {f}: {100*n_mean/total:.1f}% meaningful ({n_mean}/{total}), raw FDR-sig: {100*n_sig/total:.1f}%")
print()
print("⚠️  REMINDER: This is a DEMONSTRATION on ~40 prompts.")
print("   The real run used 480 prompts and took ~12.9h on CPU.")
print("   These numbers will NOT match docs/FINDINGS.md exactly.")
print("   See docs/METHOD.md for the full statistical method.")